# ipynb/rgcnformer_loc.ipynb - RGCNFormer 定位结果 / RGCNFormer localization results

## 项目背景 / Background
RGCNFormer 定位结果
RGCNFormer localization results

## 功能模块 / Modules
- RGCNFormer 定位结果详细展示
- (详见各代码单元 / see code cells)

## 输入 / Inputs
- 上一阶段产物(.npy/.pt/.csv/.json)/ prior-stage outputs
- 内嵌常量与参数 / inline constants and params

## 输出 / Outputs
- 图表(内联显示) / figures (inline)
- 中间变量 / intermediate variables
- 导出文件(.png/.pdf/.csv) / exported files

## 数据流 / Data Flow
1. 加载数据 / Load data
2. 运行分析 / Run analysis
3. 渲染图表 / Render figures
4. 导出 / Export

## 相关文件 / Related Files
- 调用 / Calls: inference_modx_segmented.py、ipynb/loc_compare*.ipynb
- 被调用 / Called by: 报告 / 论文 / report / paper

## 使用示例 / Usage Example
- 在 JupyterLab 中打开 / open in JupyterLab
- 逐单元运行 / run cells sequentially

## 作者 / Author
项目组 / Project Team

## 版本 / Version
1.0



In [1]:
# install.packages("ggforce")

In [4]:
library(showtext)
library(ggplot2)
library(dplyr)
library(tidyr)
library(scales)

# 1. 字体与环境配置
font_add("YaHei", 
         regular = "/usr/share/fonts/truetype/myfonts/msyh.ttf", 
         bold = "/usr/share/fonts/truetype/myfonts/msyhbd.ttf")
showtext_auto()
showtext_opts(dpi = 300)

# 2. 数据加载
df_loc <- read.csv("data/rgcnformer_loc.csv")
df_stat <- read.csv("data/statistic_loc.csv")

# 定义 K 档位映射逻辑
k_levels <- c("Top.1", "Top.3", "Top.5", "Top.7", "Top.10", "Top.20", "Top.50")
k_values <- c(1, 3, 5, 7, 10, 20, 50)
map_to_k <- function(val) {
  idx <- which(k_values >= val)[1]
  if(is.na(idx)) idx <- length(k_values)
  return(k_levels[idx])
}

# 3. 自定义函数：生成圆角矩形路径 (圆角15px效果，对应r=0.15)
generate_rounded_rect <- function(x_center, y_center, w=0.94, h=0.94, r=0.15, n=20) {
  theta <- seq(0, pi/2, length.out = n)
  tr <- data.frame(x = x_center + w/2 - r + r*cos(theta), y = y_center + h/2 - r + r*sin(theta))
  tl <- data.frame(x = x_center - w/2 + r - r*sin(theta), y = y_center + h/2 - r + r*cos(theta))
  bl <- data.frame(x = x_center - w/2 + r - r*cos(theta), y = y_center - h/2 + r - r*sin(theta))
  br <- data.frame(x = x_center + w/2 - r + r*sin(theta), y = y_center - h/2 + r - r*cos(theta))
  return(rbind(tr, tl, bl, br))
}

# 4. 构造数据层
colors_map <- c(
  "Mean"   = "#0077BB", # 清晰蓝
  "Median" = "#EE7733", # 活力橙
  "Mode"   = "#33BBEE"  # 亮青色
)
classes <- unique(df_loc$Name)
class_levels <- rev(classes)

# 圆角背景多边形数据
bg_rounded <- do.call(rbind, lapply(1:length(classes), function(i) {
  do.call(rbind, lapply(1:length(k_levels), function(j) {
    res <- generate_rounded_rect(j, i)
    res$Class <- class_levels[i]
    res$Metric <- k_levels[j]
    res$ID <- paste(i, j, sep="_")
    res
  }))
}))

# 5. 构造渐变统计竖条
cell_stats <- df_stat %>% rename(Name = Class) %>%
  mutate(Mean_K = sapply(Mean, map_to_k), Median_K = sapply(Median, map_to_k), Mode_K = sapply(Mode, map_to_k)) %>%
  select(Name, Mean_K, Median_K, Mode_K) %>%
  pivot_longer(cols = -Name, names_to = "StatType", values_to = "Metric") %>%
  mutate(StatType = gsub("_K", "", StatType))

n_segments <- 25
gradient_segments <- list()
for (cls in classes) {
  for (met in k_levels) {
    stats_in_cell <- cell_stats %>% filter(Name == cls, Metric == met) %>%
      mutate(Order = factor(StatType, levels = c("Mean", "Median", "Mode"))) %>%
      arrange(Order) %>% pull(StatType) %>% unique()
    if (length(stats_in_cell) == 0) next
    
    x_idx <- which(k_levels == met)
    y_idx <- which(class_levels == cls)
    x_seq <- seq(0, 1, length.out = n_segments)
    
    if (length(stats_in_cell) == 1) {
      base_col <- colors_map[stats_in_cell[1]]
      cols <- rep(base_col, n_segments); alphas <- 0.8 * (1 - 2 * abs(x_seq - 0.5))
    } else {
      cols <- colorRampPalette(colors_map[stats_in_cell])(n_segments); alphas <- rep(0.65, n_segments)
    }
    
    for (i in 1:(n_segments-1)) {
      gradient_segments[[length(gradient_segments) + 1]] <- data.frame(
        xmin = x_idx - 0.45 + x_seq[i]*0.9, xmax = x_idx - 0.45 + x_seq[i+1]*0.9,
        ymin = y_idx - 0.45, ymax = y_idx + 0.45,
        fill = cols[i], alpha = alphas[i]
      )
    }
  }
}
df_gradient <- do.call(rbind, gradient_segments)

# 6. 核心修改：性能主数据预处理
# 手动将低于 0.40 的数值限制为 0.40，以实现“相对气泡大小”并避免 oob 错误
plot_data <- df_loc %>%
  pivot_longer(cols = starts_with("Top"), names_to = "Metric", values_to = "Value") %>%
  mutate(
    Metric = factor(Metric, levels = k_levels),
    Class = factor(Name, levels = class_levels),
    # 创建专门用于气泡大小的列，确保不低于 0.40
    SizeValue = pmax(Value, 0.40)
  )

# 7. 绘图
p <- ggplot() +
  # A. 背景十字网格分割线 (X轴垂直线 + Y轴水平线)
  # geom_vline(xintercept = seq(0.5, length(k_levels) + 0.5, 1), color = "#EBEDEF", size = 0.6) +
  geom_hline(yintercept = seq(0.5, length(classes) + 0.5, 1), color = "#EBEDEF", size = 0.6) +
  
  # B. 圆角矩形单元格背景
  geom_polygon(data = bg_rounded, aes(x=x, y=y, group=ID), fill = "#FDFDFD", color = NA) +
  
  # C. 统计渐变竖条层
  geom_rect(data = df_gradient, aes(xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax, fill=fill, alpha=alpha), show.legend = FALSE) +
  scale_fill_identity() +
  scale_alpha_identity() +
  
  # D. 统计指标图例占位 (通过 color 生成图例)
  geom_point(data = data.frame(M=k_levels[1], C=classes[1], S=factor(names(colors_map), levels=names(colors_map))),
             aes(x=M, y=C, color=S), alpha=0) +
  scale_color_manual(name = "Statistic Distribution (Threshold Markers)", values = colors_map) +
  
  # E. 性能气泡层 (莫兰迪深色系 + 描边增强区分度)
  geom_point(data = plot_data, 
             aes(x = Metric, y = Class, size = SizeValue), 
             fill = "#4A4E69",    # 莫兰迪深紫灰，非常有高级感且能压住背景
             color = "#F2F2F2",   # 浅灰色描边，比纯白更符合莫兰迪调性
             shape = 21,          # 21号形状才支持 fill 和 color
             stroke = 0.6,        # 描边粗细
             alpha = 0.9) +
  
  # F. 数值标签 (为了在深色气泡中清晰，使用极简浅色)
  geom_text(data = plot_data, 
            aes(x = Metric, y = Class, label = sprintf("%.2f", Value * 100)),
            size = 3.2, 
            fontface = "bold", 
            family = "YaHei", 
            color = "#FDFDFD") + # 纯净文字颜色
  
  # G. 相对比例尺配置 (移除 oob 参数以解决错误)
  scale_size_continuous(
    range = c(10, 15),       # 最小和最大气泡的视觉尺寸
    limits = c(0.40, 1.0),  # 将 68% 映射为最小，100% 映射为最大
    guide = "none"
  ) +
  
  scale_x_discrete(position = "top", labels = gsub("Top.", "Top-", k_levels), expand = c(0.02, 0.02)) +
  scale_y_discrete(expand = c(0.05, 0.05)) +
  theme_minimal() +
  theme(
    text = element_text(family = "YaHei"),
    axis.title = element_blank(),
    axis.text = element_text(size = 11, face = "bold", color = "#4A4A4A"),
    panel.grid = element_blank(), 
    legend.position = "bottom",
    legend.box.background = element_rect(color="#D5D8DC", fill="#FBFCFC", size=0.5),
    legend.margin = margin(6, 10, 6, 10),
    plot.margin = margin(15, 15, 15, 15)
  ) +
  guides(color = guide_legend(override.aes = list(alpha=1, size=5, shape=15), direction = "horizontal", title.position = "top"))

# 8. 保存结果
ggsave("png/localization_recall_relative_grid.png", p, width = 12, height = 10, dpi = 300)

In [3]:
library(showtext)
library(ggplot2)
library(dplyr)
library(tidyr)
library(scales)
library(ggforce)

# 1. 字体与环境配置
font_add("YaHei", 
         regular = "/usr/share/fonts/truetype/myfonts/msyh.ttf", 
         bold = "/usr/share/fonts/truetype/myfonts/msyhbd.ttf")
showtext_auto()
showtext_opts(dpi = 300)

# 2. 数据加载
df_loc <- read.csv("data/rgcnformer_loc.csv")
df_stat <- read.csv("data/statistic_loc.csv")

# 定义 K 档位映射逻辑
k_levels <- c("Top.1", "Top.3", "Top.5", "Top.7", "Top.10", "Top.20", "Top.50")
k_values <- c(1, 3, 5, 7, 10, 20, 50)
map_to_k <- function(val) {
  idx <- which(k_values >= val)[1]
  if(is.na(idx)) idx <- length(k_values)
  return(k_levels[idx])
}

# 3. 自定义函数：生成单元格圆角背景
generate_rounded_rect <- function(x_center, y_center, w=0.94, h=0.94, r=0.15, n=20) {
  theta <- seq(0, pi/2, length.out = n)
  tr <- data.frame(x = x_center + w/2 - r + r*cos(theta), y = y_center + h/2 - r + r*sin(theta))
  tl <- data.frame(x = x_center - w/2 + r - r*sin(theta), y = y_center + h/2 - r + r*cos(theta))
  bl <- data.frame(x = x_center - w/2 + r - r*cos(theta), y = y_center - h/2 + r - r*sin(theta))
  br <- data.frame(x = x_center + w/2 - r + r*sin(theta), y = y_center - h/2 + r - r*cos(theta))
  return(rbind(tr, tl, bl, br))
}

# 4. 构造背景数据层
classes <- unique(df_loc$Name)
class_levels <- rev(classes)

bg_rounded <- do.call(rbind, lapply(1:length(classes), function(i) {
  do.call(rbind, lapply(1:length(k_levels), function(j) {
    res <- generate_rounded_rect(j, i)
    res$Class <- class_levels[i]
    res$Metric <- k_levels[j]
    res$ID <- paste(i, j, sep="_")
    res
  }))
}))

# 5. 构造统计渐变竖条层
colors_map <- c("Mean" = "#0077BB", "Median" = "#EE7733", "Mode" = "#33BBEE")
cell_stats <- df_stat %>% rename(Name = Class) %>%
  mutate(Mean_K = sapply(Mean, map_to_k), Median_K = sapply(Median, map_to_k), Mode_K = sapply(Mode, map_to_k)) %>%
  select(Name, Mean_K, Median_K, Mode_K) %>%
  pivot_longer(cols = -Name, names_to = "StatType", values_to = "Metric") %>%
  mutate(StatType = gsub("_K", "", StatType))

n_segments <- 25
gradient_segments <- list()
for (cls in classes) {
  for (met in k_levels) {
    stats_in_cell <- cell_stats %>% filter(Name == cls, Metric == met) %>%
      mutate(Order = factor(StatType, levels = c("Mean", "Median", "Mode"))) %>%
      arrange(Order) %>% pull(StatType) %>% unique()
    if (length(stats_in_cell) == 0) next
    
    x_idx <- which(k_levels == met)
    y_idx <- which(class_levels == cls)
    x_seq <- seq(0, 1, length.out = n_segments)
    
    if (length(stats_in_cell) == 1) {
      base_col <- colors_map[stats_in_cell[1]]
      cols <- rep(base_col, n_segments); alphas <- 0.8 * (1 - 2 * abs(x_seq - 0.5))
    } else {
      cols <- colorRampPalette(colors_map[stats_in_cell])(n_segments); alphas <- rep(0.65, n_segments)
    }
    
    for (i in 1:(n_segments-1)) {
      gradient_segments[[length(gradient_segments) + 1]] <- data.frame(
        xmin = x_idx - 0.45 + x_seq[i]*0.9, xmax = x_idx - 0.45 + x_seq[i+1]*0.9,
        ymin = y_idx - 0.45, ymax = y_idx + 0.45,
        fill = cols[i], alpha = alphas[i]
      )
    }
  }
}
df_gradient <- do.call(rbind, gradient_segments)

# 6. 性能数据预处理
plot_data <- df_loc %>%
  pivot_longer(cols = starts_with("Top"), names_to = "Metric", values_to = "Value") %>%
  mutate(
    Metric_num = as.numeric(factor(Metric, levels = k_levels)),
    Class_num = as.numeric(factor(Name, levels = class_levels)),
    start_angle = 0,
    end_angle = Value * 2 * pi
  )

# 7. 绘图
p <- ggplot() +
  # A. 背景
  geom_hline(yintercept = seq(0.5, length(classes) + 0.5, 1), color = "#EBEDEF", size = 0.6) +
  geom_polygon(data = bg_rounded, aes(x=x, y=y, group=ID), fill = "#FDFDFD", color = NA) +
  
  # B. 统计渐变竖条
  geom_rect(data = df_gradient, aes(xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax, fill=fill, alpha=alpha), show.legend = FALSE) +
  
  # C. 进度环底层轨道 (调薄：linewidth = 2.5)
  geom_arc(data = plot_data, 
           aes(x0 = Metric_num, y0 = Class_num, r = 0.30, start = 0, end = 2*pi),
           color = "#E8EAEB", linewidth = 2.5, alpha = 0.8) +
  
  # D. 进度环进度层 (调薄：linewidth = 1.5，带圆角)
  geom_arc(data = plot_data, 
           aes(x0 = Metric_num, y0 = Class_num, r = 0.30, 
               start = start_angle, end = end_angle),
           color = "#4A4E69", linewidth = 1.5, lineend = "round") +
  
  # E. 数值标签
  geom_text(data = plot_data, 
            aes(x = Metric_num, y = Class_num, label = sprintf("%.0f", Value * 100)),
            size = 3.2, fontface = "bold", family = "YaHei", color = "#4A4E69") + 
  
  # F. 核心配置：锁定正圆
  coord_fixed(ratio = 1) + 
  
  # G. 比例尺与图例修复
  scale_fill_identity() +
  scale_alpha_identity() +
  scale_x_continuous(breaks = 1:length(k_levels), labels = gsub("Top.", "Top-", k_levels), 
                     position = "top", expand = c(0.02, 0.02)) +
  scale_y_continuous(breaks = 1:length(classes), labels = class_levels, expand = c(0.05, 0.05)) +
  
  geom_point(data = data.frame(M=1, C=1, S=factor(names(colors_map), levels=names(colors_map))),
             aes(x=M, y=C, color=S), alpha=0) +
  scale_color_manual(name = "Statistic\nDistribution", values = colors_map) +
  
  theme_minimal() +
  theme(
    text = element_text(family = "YaHei"),
    axis.title = element_blank(),
    axis.text = element_text(size = 11, face = "bold", color = "#4A4A4A"),
    panel.grid = element_blank(), 
    # --- 关键修改：图例位置设置 ---
    legend.position = c(1.25, 0.1),             # 坐标 (1,0) 代表右下角
    legend.justification = c(1, 0),        # 锚点对齐右下角
    legend.direction = "vertical",         # 强制竖排
    legend.background = element_rect(color="#D5D8DC", fill="#FBFCFC", size=0.5),
    legend.margin = margin(6, 6, 6, 6),
    # ----------------------------
    plot.margin = margin(15, 60, 15, 15)   # 增加右侧边距 (60)，防止图例超出边界
  ) +
  guides(color = guide_legend(
    override.aes = list(alpha = 1, size = 5, shape = 15), 
    title.position = "top",
    ncol = 1                               # 确保只有 1 列，实现严格竖排
  ))

# 8. 保存结果
ggsave("png/localization_recall_thin_donut.pdf", p, width = 12, height = 10, dpi = 300)

Warning message:
"Using `size` aesthetic for lines was deprecated in ggplot2 3.4.0.
i Please use `linewidth` instead."
Warning message:
"The `size` argument of `element_rect()` is deprecated as of ggplot2 3.4.0.
i Please use the `linewidth` argument instead."
